# Consumer Segmentation & Lifetime Value

## Objetivos de Aprendizaje

En este notebook, aprenderás a learn how to:

1. **Segment customers** using RFM analysis with `NTILE()`
2. **Calculate lifetime value** with `SNOWFLAKE.ML.FORECAST`
3. **Score RFM dimensions** (Recency, Frequency, Monetary)
4. **Assign segments** using business rules (Champions, At-Risk, Lost)
5. **Forecast 12-month LTV** for each segment

---

## What You'll Build

A **customer segmentation system** that predicts lifetime value:
- RFM scoring using SQL quintiles
- Customer segments (Champions, Loyal, At-Risk, Lost)
- LTV forecasting with ML
- Segment-specific targeting strategies
- Marketing optimization recommendations

---

## Technical Requirements

- **Snowflake account** with standard SQL access
- **Approximately 15 minutes** to complete
- **100% SQL** (except Streamlit dashboard)

---

## Key Snowflake Features Used

- `NTILE(5)` - Score RFM dimensions into quintiles
- `SNOWFLAKE.ML.FORECAST` - Predict lifetime value
- `CASE` statements - Business rule segmentation
- `AVG() OVER()` - Calculate segment averages
- `DATEDIFF()` - Recency calculations

Let's begin!

---

## Paso 1: Configuración del Entorno

### RFM Segmentation

**R**ecency: Days since last purchase  
**F**requency: Number of purchases  
**M**onetary: Total spend

Segments: Champions, Loyal, At-Risk, Lost

In [ ]:
CREATE DATABASE IF NOT EXISTS CUSTOMER_SEGMENTATION_DB;
USE SCHEMA CUSTOMER_SEGMENTATION_DB.PUBLIC;
CREATE WAREHOUSE IF NOT EXISTS COMPUTE_WH WITH WAREHOUSE_SIZE='SMALL';
USE WAREHOUSE COMPUTE_WH;

SELECT CURRENT_DATABASE() as db;

---

## Paso 2: Create Tables

In [ ]:
CREATE OR REPLACE TABLE customers (
    customer_id VARCHAR(50) PRIMARY KEY,
    signup_date DATE,
    customer_type VARCHAR(50)
);

CREATE OR REPLACE TABLE purchases (
    purchase_id VARCHAR(50) PRIMARY KEY,
    customer_id VARCHAR(50),
    purchase_date DATE,
    purchase_amount DECIMAL(10,2)
);

---

## Paso 3: Generar Customer & Purchase Data

In [ ]:
-- Generar 1000 customers
INSERT INTO customers
SELECT 
    'CUST' || LPAD(SEQ4(), 5, '0') as customer_id,
    DATEADD('day', -FLOOR(UNIFORM(30, 730, RANDOM())), CURRENT_DATE()) as signup_date,
    CASE FLOOR(UNIFORM(1, 4, RANDOM()))
        WHEN 1 THEN 'Patient'
        WHEN 2 THEN 'Caregiver'
        ELSE 'Healthcare Professional'
    END as customer_type
FROM TABLE(GENERATOR(ROWCOUNT => 1000));

-- Generar purchases (varying frequency per customer)
INSERT INTO purchases
SELECT 
    UUID_STRING() as purchase_id,
    'CUST' || LPAD(FLOOR(UNIFORM(1, 1001, RANDOM())), 5, '0') as customer_id,
    DATEADD('day', -FLOOR(UNIFORM(1, 365, RANDOM())), CURRENT_DATE()) as purchase_date,
    ROUND(UNIFORM(50, 500, RANDOM()), 2) as purchase_amount
FROM TABLE(GENERATOR(ROWCOUNT => 5000));

SELECT COUNT(DISTINCT customer_id) as customers, COUNT(*) as purchases
FROM purchases;

---

## Paso 4: Calculate RFM Scores

Use SQL `NTILE()` para puntuar each dimension (1-5 scale)

In [ ]:
-- Calculate RFM metrics
CREATE OR REPLACE TABLE rfm_analysis AS
WITH customer_metrics AS (
    SELECT 
        c.customer_id,
        c.customer_type,
        DATEDIFF('day', MAX(p.purchase_date), CURRENT_DATE()) as recency_days,
        COUNT(p.purchase_id) as frequency,
        SUM(p.purchase_amount) as monetary_value,
        AVG(p.purchase_amount) as avg_order_value
    FROM customers c
    LEFT JOIN purchases p ON c.customer_id = p.customer_id
    GROUP BY c.customer_id, c.customer_type
),
rfm_scores AS (
    SELECT 
        customer_id,
        customer_type,
        recency_days,
        frequency,
        monetary_value,
        avg_order_value,
        -- Score 1-5 (5 = best)
        NTILE(5) OVER (ORDER BY recency_days DESC) as recency_score,  -- Lower recency = better
        NTILE(5) OVER (ORDER BY frequency) as frequency_score,
        NTILE(5) OVER (ORDER BY monetary_value) as monetary_score
    FROM customer_metrics
)
SELECT 
    *,
    (recency_score + frequency_score + monetary_score) / 3.0 as rfm_score,
    -- Segment based on RFM
    CASE 
        WHEN rfm_score >= 4.5 THEN '🏆 Champions'
        WHEN rfm_score >= 3.5 THEN '⭐ Loyal Customers'
        WHEN rfm_score >= 2.5 THEN '👤 Potential Loyalists'
        WHEN rfm_score >= 1.5 AND recency_score >= 3 THEN '⚠️ At Risk'
        ELSE '😴 Lost'
    END as segment
FROM rfm_scores;

-- View segment distribution
SELECT segment, COUNT(*) as customer_count, 
       ROUND(AVG(monetary_value), 0) as avg_value,
       ROUND(AVG(frequency), 1) as avg_frequency
FROM rfm_analysis
GROUP BY segment
ORDER BY avg_value DESC;

---

## Paso 5: Forecast Lifetime Value

Use `CREATE SNOWFLAKE.ML.FORECAST` to predict future customer spend

### Process

**1. Preparar Datos de Entrenamiento** - Aggregate monthly spend by cohort  
**2. Create Training View** - Extract columns needed for forecasting  
**3. Train Model** - Reference view in `SYSTEM$REFERENCE`

### Key Points

-  Create a view from your table for training data
-  Use `SYSTEM$REFERENCE('VIEW', 'view_name')` to reference it
-  Cannot pass inline SELECT queries to SYSTEM$REFERENCE

In [ ]:
-- Prepare monthly cohort spend for forecasting
CREATE OR REPLACE TABLE monthly_cohort_spend AS
SELECT 
    DATE_TRUNC('month', p.purchase_date) as spend_month,
    DATE_TRUNC('month', c.signup_date) as cohort_month,
    COUNT(DISTINCT p.customer_id) as active_customers,
    SUM(p.purchase_amount) as total_spend
FROM purchases p
JOIN customers c ON p.customer_id = c.customer_id
GROUP BY spend_month, cohort_month
ORDER BY cohort_month, spend_month;

-- Create view for forecast training (aggregate by month)
CREATE OR REPLACE VIEW ltv_training_data AS
SELECT spend_month, total_spend 
FROM monthly_cohort_spend;

-- Train LTV forecast model
CREATE OR REPLACE SNOWFLAKE.ML.FORECAST ltv_forecast(
    INPUT_DATA => SYSTEM$REFERENCE('VIEW', 'ltv_training_data'),
    TIMESTAMP_COLNAME => 'spend_month',
    TARGET_COLNAME => 'total_spend'
);

SELECT 'LTV forecast model trained!' as status;

---

## Paso 6: Generar LTV Predictions

In [ ]:
-- Forecast next 12 months
CREATE OR REPLACE TABLE ltv_predictions AS
SELECT 
    ts as forecast_month,
    forecast as predicted_spend,
    lower_bound as conservative_spend,
    upper_bound as optimistic_spend
FROM TABLE(ltv_forecast!FORECAST(FORECASTING_PERIODS => 12));

-- Calculate LTV by segment
CREATE OR REPLACE VIEW segment_ltv AS
WITH current_value AS (
    SELECT segment, SUM(monetary_value) as current_total
    FROM rfm_analysis
    GROUP BY segment
),
forecast_value AS (
    SELECT SUM(predicted_spend) as forecast_total
    FROM ltv_predictions
)
SELECT 
    cv.segment,
    cv.current_total,
    -- Allocate forecast proportionally
    ROUND((cv.current_total / SUM(cv.current_total) OVER ()) * fv.forecast_total, 0) as predicted_12mo_value,
    ROUND(predicted_12mo_value / NULLIF(COUNT(*) OVER (PARTITION BY cv.segment), 0), 0) as avg_ltv_per_customer
FROM current_value cv
CROSS JOIN forecast_value fv
JOIN rfm_analysis ra ON cv.segment = ra.segment
GROUP BY cv.segment, cv.current_total, fv.forecast_total;

SELECT * FROM segment_ltv ORDER BY avg_ltv_per_customer DESC;

---

## Paso 7: Interactive Segmentation Dashboard

In [ ]:
import streamlit as st
import pandas as pd
from snowflake.snowpark.context import get_active_session

session = get_active_session()

st.title("👥 Customer Segmentation & LTV")

# Segment distribution
segments = session.sql("""
    SELECT segment, COUNT(*) as count, 
           ROUND(AVG(monetary_value),0) as avg_value
    FROM rfm_analysis GROUP BY segment
""").to_pandas()

col1, col2 = st.columns(2)
with col1:
    champions = segments[segments['SEGMENT']=='🏆 Champions']['COUNT'].sum() if len(segments[segments['SEGMENT']=='🏆 Champions']) > 0 else 0
    st.metric("Champions", int(champions))
with col2:
    at_risk = segments[segments['SEGMENT']=='⚠️ At Risk']['COUNT'].sum() if len(segments[segments['SEGMENT']=='⚠️ At Risk']) > 0 else 0
    st.metric("At Risk", int(at_risk))

st.subheader("📊 Segment Distribution")
st.bar_chart(segments.set_index('SEGMENT')['COUNT'])

# LTV by segment
st.subheader("💰 Lifetime Value by Segment")
ltv = session.sql("SELECT * FROM segment_ltv ORDER BY avg_ltv_per_customer DESC").to_pandas()
st.dataframe(ltv, use_container_width=True, hide_index=True)

# RFM scatter
st.subheader("🎯 RFM Analysis")
rfm = session.sql("""
    SELECT segment, recency_days, frequency, monetary_value
    FROM rfm_analysis LIMIT 500
""").to_pandas()

st.scatter_chart(rfm, x='FREQUENCY', y='MONETARY_VALUE', color='SEGMENT')

# Forecast
st.subheader("📈 12-Month LTV Forecast")
forecast = session.sql("""
    SELECT forecast_month, predicted_spend, 
           conservative_spend, optimistic_spend
    FROM ltv_predictions ORDER BY forecast_month
""").to_pandas()

st.line_chart(forecast.set_index('FORECAST_MONTH'))

---

##  Complete!

### What You Learned

1.  RFM segmentation with SQL (`NTILE()` function)
2.  Customer segmentation using business rules
3.  LTV forecasting with `SNOWFLAKE.ML.FORECAST`
4.  Cohort analysis and prediction
5.  Segmentation dashboards

### RFM Scoring

```sql
-- Score 1-5 (5 = best)
NTILE(5) OVER (ORDER BY recency_days DESC) as recency_score
NTILE(5) OVER (ORDER BY frequency) as frequency_score
NTILE(5) OVER (ORDER BY monetary_value) as monetary_score
```

### Segments

- ** Champions**: Alto RFM (4.5+)
- ** Loyal**: Good RFM (3.5-4.5)
- ** At Risk**: Decreasing engagement
- ** Lost**: No recent activity

### Applications

- Personalized marketing campaigns
- Retention strategies for at-risk customers
- VIP programs for champions
- Win-back campaigns for lost customers

[Snowflake NTILE Docs](https://docs.snowflake.com/en/sql-reference/functions/ntile)

---

## Limpieza del Entorno (Opcional)

### Qué Hace Esto

This cell will **completely remove** all objects created in this tutorial:
- Drops the `CUSTOMER_SEGMENTATION_DB` database (and all tables/models within it)
- Drops the `COMPUTE_WH` warehouse

### When to Use

Run this if you want to:
- Clean up your Snowflake environment after completing the tutorial
- Start fresh and re-run the entire notebook
- Remove all demo data

### Instructions

**This cell is commented out by default** to prevent accidental deletion when running "Run All".

To reset your environment:
1. **Remove the comment markers** (`--`) from the SQL commands below
2. **Run this cell manually**

 **Warning**: This action cannot be undone. All data and models will be permanently deleted.

In [ ]:
-- Descomentar las líneas siguientes and ejecutar esta celda para limpiar el entorno

-- DROP DATABASE IF EXISTS CUSTOMER_SEGMENTATION_DB;
-- DROP WAREHOUSE IF EXISTS COMPUTE_WH;
